In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
URL_FULL = "http://localhost:8082/"
URL_QUANTIZED = "http://localhost:8081/"
MAX_TOKENS = 30
TEMPERATURE = 0.99
SEED = 42
LOGPROBS = 4


from validation.prompts import get_squad_data_questions
from validation.runner import run_validation
from validation.data import (
    ModelInfo,
    RequestParams,
    save_to_jsonl
)


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-235B-A22B-Instruct-2507-FP8")

In [3]:
from datasets import load_dataset
from typing import List


def get_squad_data_questions() -> List[str]:
    dataset = load_dataset('squad', keep_in_memory=True)
    prompts = []
    
    train_prompts = [f"Context: {context}\nQuestion: {question} " for question, context in zip(dataset['train']['question'], dataset['train']['context'])]
    prompts.extend(train_prompts)
    
    validation_prompts = [f"Context: {context}\nQuestion: {question} " for question, context in zip(dataset['validation']['question'], dataset['validation']['context'])]
    prompts.extend(validation_prompts)
    
    return prompts

prompts = get_squad_data_questions()

In [4]:
full_model_info = ModelInfo(
    url=URL_FULL,
    name="Qwen/Qwen3-235B-A22B-Instruct-2507-FP8",
    deploy_params={
        "GPU": "8xH100",
        "precision": "fp8",
    }
)

quantized_model_info = ModelInfo(
    url=URL_QUANTIZED,
    name="Qwen/Qwen3-235B-A22B-Instruct-2507-FP8",
    deploy_params={
        "GPU": "4xPRO6000",
        "precision": "fp8",
    }
)

request_params = RequestParams(
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
    top_logprobs=LOGPROBS
)

inference_model_info = quantized_model_info
validation_model_info = full_model_info

In [7]:
from validation.utils import inference, validation, _extract_logprobs, _extract_enforced_tokens

# Pick a sample prompt
sample_prompt = prompts[0]
print(f"Prompt: {sample_prompt[:20]}...")

# Send inference request to the full model
print("\n--- Inference (full model) ---")
inference_resp = inference(
    inference_model_info,
    request_params,
    sample_prompt,
)
inference_result = _extract_logprobs(inference_resp)
enforced_tokens = _extract_enforced_tokens(inference_resp)

print(f"Response text: {inference_result.text[:500]}...")
print(f"Number of tokens: {len(inference_result.results)}")
print(f"First 5 logprobs: {[r.logprobs for r in inference_result.results[:5]]}")

Prompt: Context: Architectur...

--- Inference (full model) ---
Response text: The Virgin Mary allegedly appeared to **Saint Bernadette Soubirous** in 1858 in Lourdes, France....
Number of tokens: 30
First 5 logprobs: [{'The': -2.3841855067985307e-07, 'In': -15.40625, 'According': -17.15625, 'To': -19.484375}, {' Virgin': -2.3841855067985307e-07, 'Virgin': -15.4375, ' virgin': -18.484375, ' Virginia': -18.671875}, {' Mary': 0.0, 'Mary': -19.203125, ' Maria': -19.5625, ' M': -19.6875}, {' allegedly': -5.280832192511298e-05, ' reportedly': -10.156302452087402, ' was': -11.500052452087402, ' is': -12.734427452087402}, {' appeared': 0.0, 'appeared': -17.71875, ' appearing': -19.40625, ' appe': -19.78125}]


In [11]:
# Test quantized model directly (without enforced tokens)
print("--- Quantized model inference ---")
quantized_resp = inference(
    full_model_info,
    request_params,
    sample_prompt,
)
quantized_result = _extract_logprobs(quantized_resp)
enforced_tokens = _extract_enforced_tokens(quantized_resp)

print(f"Response text: {quantized_result.text[:500]}...")
print(f"Number of tokens: {len(quantized_result.results)}")
print(f"First 5 logprobs: {[r.logprobs for r in quantized_result.results[:5]]}")

--- Quantized model inference ---
Response text: The Virgin Mary allegedly appeared to **Saint Bernadette Soubirous** in 1858 in Lourdes, France....
Number of tokens: 30
First 5 logprobs: [{'785': 0.0, '2': -9999.0, '0': -9999.0, '1': -9999.0}, {'11214': 0.0, '2': -9999.0, '0': -9999.0, '1': -9999.0}, {'10244': 0.0, '2': -9999.0, '0': -9999.0, '1': -9999.0}, {'19204': 0.0, '2': -9999.0, '0': -9999.0, '1': -9999.0}, {'9723': 0.0, '2': -9999.0, '0': -9999.0, '1': -9999.0}]


In [12]:
DATA_PATH = 'pro6000.jsonl'

batch_size = 50

prompts = prompts[:10]

for start_idx in range(0, len(prompts), batch_size):
    prompt_batch = prompts[start_idx:start_idx + batch_size]
    results_batch = run_validation(
        prompt_batch,
        inference_model=inference_model_info,
        validation_model=validation_model_info,
        request_params=request_params,
        max_workers=50
    )
    save_to_jsonl(results_batch, DATA_PATH, append=True)
    print(f"Processed {start_idx + batch_size} from {len(prompts)}")

RuntimeError: Validation API request failed with status 400 {"object":"error","message":"invalid literal for int() with base 10: 'The'","type":"BadRequestError","param":null,"code":400}
(enforced_tokens: tokens=[EnforcedToken(token='The', top_tokens=['The', 'Answer', 'There', 'the']), EnforcedToken(token=' daily', top_tokens=[' daily', ' Daily', ' weekly', 'daily']), EnforcedToken(token=' student', top_tokens=[' student', ' undergraduate', ' user', ' Student']), EnforcedToken(token=' paper', top_tokens=[' paper', ' newspaper', '纸', '.paper']), EnforcedToken(token=' at', top_tokens=[' at', 'at', ' is', ' tại']), EnforcedToken(token=' Notre', top_tokens=[' Notre', ' the', ' University', ' Not']), EnforcedToken(token=' Dame', top_tokens=[' Dame', ' dame', ' Dam', ' Dane']), EnforcedToken(token=' is', top_tokens=[' is', ' called', ',', ' was']), EnforcedToken(token=' called', top_tokens=[' called', 'called', ' calle', '.called']), EnforcedToken(token=' **', top_tokens=[' **', ' *', ' The',